In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    output_hidden_states=True
)

model.eval()

c:\Users\anush\anaconda3main\envs\steering\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 453/453 [00:05<00:00, 80.65it/s, Materializing param=model.layers.31.self_attn.v_proj.weight] 
Some parameters are on the meta device because they were offloaded to the cpu.


PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2560)
    (layers): ModuleList(
      (0-31): 32 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (k_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (v_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (dense): Linear(in_features=2560, out_features=2560, bias=True)
        )
        (mlp): PhiMLP(
          (activation_fn): NewGELUActivation()
          (fc1): Linear(in_features=2560, out_features=10240, bias=True)
          (fc2): Linear(in_features=10240, out_features=2560, bias=True)
        )
        (input_layernorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (rotary_emb): PhiRotaryEmbedding()
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (final_layernorm): LayerNorm((2560,), eps=1

In [3]:
import json

with open("explanation_dataset_clean.json", "r") as f:
    clean_dataset = json.load(f)

print(len(clean_dataset))

491


In [4]:
import numpy as np

def get_representation(text, layer=16):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        outputs = model(**inputs)

    hidden_states = outputs.hidden_states[layer]  # (batch, seq_len, dim)

    # Mean pool over tokens
    pooled = hidden_states.mean(dim=1)

    return pooled.squeeze().cpu().numpy()

In [5]:
test_subset = clean_dataset[:5]

elem_reps = []
adv_reps = []

for item in test_subset:
    elem_reps.append(get_representation(item["elementary"], layer=16))
    adv_reps.append(get_representation(item["advanced"], layer=16))

elem_reps = np.vstack(elem_reps)
adv_reps = np.vstack(adv_reps)

print(elem_reps.shape)

(5, 2560)


In [6]:
elem_reps = []
adv_reps = []

for idx, item in enumerate(clean_dataset):
    elem_reps.append(get_representation(item["elementary"], layer=16))
    adv_reps.append(get_representation(item["advanced"], layer=16))

    if idx % 50 == 0:
        torch.cuda.empty_cache()
        print(f"Processed {idx} samples")

elem_reps = np.vstack(elem_reps)
adv_reps = np.vstack(adv_reps)

print("Done. Shape:", elem_reps.shape)

Processed 0 samples
Processed 50 samples
Processed 100 samples
Processed 150 samples
Processed 200 samples
Processed 250 samples
Processed 300 samples
Processed 350 samples
Processed 400 samples
Processed 450 samples
Done. Shape: (491, 2560)


In [7]:
mu_elem = elem_reps.mean(axis=0)
mu_adv = adv_reps.mean(axis=0)

v_explanation = mu_adv - mu_elem
v_explanation = v_explanation / np.linalg.norm(v_explanation)

print("Vector dimension:", v_explanation.shape)

Vector dimension: (2560,)


In [8]:
proj_elem = elem_reps @ v_explanation
proj_adv = adv_reps @ v_explanation

print("Elementary mean:", proj_elem.mean())
print("Advanced mean:", proj_adv.mean())

Elementary mean: -31.47
Advanced mean: -2.754


In [9]:
import numpy as np

def cohens_d(a, b):
    return (np.mean(a) - np.mean(b)) / np.sqrt(
        (np.var(a) + np.var(b)) / 2
    )

d_proj = cohens_d(proj_adv, proj_elem)
print("Cohen's d (projection space):", d_proj)

Cohen's d (projection space): 6.98


In [11]:
import torch

v_torch = torch.tensor(v_explanation, dtype=torch.float16).to("cuda")

In [12]:
print(v_torch.shape)  # should be (2560,)

torch.Size([2560])


In [20]:
model.model.layers[16]

PhiDecoderLayer(
  (self_attn): PhiAttention(
    (q_proj): Linear(in_features=2560, out_features=2560, bias=True)
    (k_proj): Linear(in_features=2560, out_features=2560, bias=True)
    (v_proj): Linear(in_features=2560, out_features=2560, bias=True)
    (dense): Linear(in_features=2560, out_features=2560, bias=True)
  )
  (mlp): PhiMLP(
    (activation_fn): NewGELUActivation()
    (fc1): Linear(in_features=2560, out_features=10240, bias=True)
    (fc2): Linear(in_features=10240, out_features=2560, bias=True)
  )
  (input_layernorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
  (resid_dropout): Dropout(p=0.1, inplace=False)
)

In [21]:
def make_hook(alpha):
    def hook(module, input, output):
        # Phi layer returns tuple:
        # (hidden_states, present_key_value, ...)
        
        hidden_states = output[0]  # tensor (batch, seq_len, hidden_dim)
        
        steered_hidden = hidden_states + alpha * v_torch
        
        # Reconstruct full tuple
        return (steered_hidden,) + output[1:]
    
    return hook

In [22]:
def generate_with_steering(prompt, alpha=0.0, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    hook = model.model.layers[16].register_forward_hook(make_hook(alpha))

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9
        )

    hook.remove()

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [24]:
import numpy as np

np.save("v_explanation.npy", v_explanation)
np.save("mu_elem.npy", mu_elem)
np.save("mu_adv.npy", mu_adv)

print("Saved steering vectors.")

Saved steering vectors.
